In [5]:
import pandas as pd
import json
from collections import defaultdict

input_jsonl_file = "Amazon_Fashion.jsonl"
output_csv_file = "amazon_fashion_behavior_sequences.csv"

print("Step 1: Mapping user interaction frequencies across the dataset...")
user_counts = defaultdict(int)

# First pass: Count how many times each user appears
with open(input_jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        user_id = data.get('user_id')
        if user_id:
            user_counts[user_id] += 1

# Filter for active users who have at least 3 or more behavioral interactions
active_users = {user for user, count in user_counts.items() if count >= 3}
print(f"Found {len(active_users)} users with multi-step behavioral sequences.")

print("\nStep 2: Extracting full interaction journeys for active users...")
sampled_records = []

# Second pass: Extract the full history for these active users until we hit ~3,000 rows
with open(input_jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        user_id = data.get('user_id')
        
        if user_id in active_users:
            sampled_records.append(data)
            
        # Stop once we have a rich behavioral matrix of around 3,500 rows
        if len(sampled_records) >= 3500:
            break

# Convert to DataFrame
df = pd.DataFrame(sampled_records)

# Sort chronologically to rebuild the real-time clickstream
df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.sort_values(by=['user_id', 'datetime']).reset_index(drop=True)

# Engineer explicit Real-Time Behavioral Signals
df['session_interaction_count'] = df.groupby('user_id').cumcount() + 1
df['time_delta_seconds'] = df.groupby('user_id')['timestamp'].diff() / 1000.0
df['time_delta_seconds'] = df['time_delta_seconds'].fillna(-1)

# Engineer the Return Label (Y)
return_keywords = ['return', 'returned', 'refund', 'sent back', 'wrong size', 'too small', 'too large', 'mismatch']
df['text'] = df['text'].fillna('')
df['title'] = df['title'].fillna('')
keyword_match = df['text'].str.contains('|'.join(return_keywords), case=False) | \
                df['title'].str.contains('|'.join(return_keywords), case=False)
df['is_returned'] = ((df['rating'] <= 3) & keyword_match).astype(int)

# Trim down to exactly your target size and save
df_final = df.head(3000)[[
    'user_id', 'parent_asin', 'datetime', 'rating', 'title', 'text',
    'session_interaction_count', 'time_delta_seconds', 'is_returned'
]]

df_final.to_csv(output_csv_file, index=False, encoding='utf-8')
print(f"\nSuccess! Your true research dataset is saved as: {output_csv_file}")
print(f"Varying interaction steps now available to train your model!")
print(df_final['session_interaction_count'].value_counts())

Step 1: Mapping user interaction frequencies across the dataset...
Found 80681 users with multi-step behavioral sequences.

Step 2: Extracting full interaction journeys for active users...

Success! Your true research dataset is saved as: amazon_fashion_behavior_sequences.csv
Varying interaction steps now available to train your model!
session_interaction_count
1      313
2      313
3      313
4      185
5      130
      ... 
307      1
308      1
309      1
310      1
311      1
Name: count, Length: 311, dtype: int64
